# SISEPUEDE Tutorial #4 - The `SISEPUEDE` Object

Welcome to the **SImulation of SEctoral Pathways with Uncertainty Exploration for DEcarbonization (SISEPUEDE)** tutorials! This tutorial walks users through the `SISEPUEDE` object, which is a centralized management system for models, the database, strategies, and uncertainty exploration. This notebook walks users through:

1. Instantiating a SISEPUEDE object
2. Running scenarios and understanding dimensions of analysis (strategies, futures, design) 
4. Reading output data
5. Plotting/accessing variables

In [1]:
import warnings
warnings.filterwarnings("ignore")

import logging
import matplotlib.pyplot as plt
import numpy as np
import os, os.path
import pandas as pd
import sisepuede as ssp
import sisepuede.manager.sisepuede_examples as sxl
import sisepuede.visualization.plots as spp
import sisepuede.transformers as trf
import sisepuede.utilities._plotting as sup
from typing import *

log = None

##  Initialize the SISEPUEDE class to get started running some models
- see ``?SISEPUEDE`` for more information on initialization arguments
- Once transformations are defined, it's easy to 

In [2]:

path_strategies = # ENTER YOUR PATH HERE FOR OUTPUT STRATEGIES--SEE TUTORIAL 3 FOR MORE INFORMATION ON THIS
examples = sxl.SISEPUEDEExamples()
df_examples = examples("input_data_frame")

##  SET UP TRANSFORMATIONS/STRATEGIES
#
#  NOTE: This is an issue that's being worked out:
#        To instantiate a strategies object, need to build transformers
#        and transformations object with a dataframe. Use df_examples;
#        since we're not building tables, it won't be used, but it just
#        allows the object to instantiate.

transformers = trf.Transformers({}, df_input = df_examples,)
if not path_transformations.is_dir():
    trf.instantiate_default_strategy_directory(
        transformers,
        path_strategies,
    )
    
transformations = trf.Transformations(
    path_strategies,
    transformers = transformers,
)

strategies = trf.Strategies(
    transformations,
    export_path = "transformations",
)


# built SISEPUEDE object
sisepuede = ssp.SISEPUEDE(
    "calibrated",
    db_type = "csv", # include this if you want to save inputs
    logger = log,
    strategies = strategies,
    regions = [_REGION_NAME],
)

log = sisepuede.logger


SyntaxError: invalid syntax (2166465666.py, line 1)

###  Call the .project_scenarios() method (or simply call the SISEPUEDE object) to write outputs directly to a database (prevents significant memory usage)
- This method returns a list of primary keys that were successfully run
- The first positional argument, ``primary_keys``, can be a list of primary keys *or* a dictionary of scenario dimensions
    - e.g., ``sisepuede.project_scenarios([0, 5, 1989])`` uses 3 primary keys
    - ``sisepuede.project_scenarios({"strategy_id": [0], "future_id": [0, 9, 903]})`` specifies a scenario dimensional subset of primary keys
- see ``?sisepuede.project_scenarios`` for more information on inputs



In [ ]:
sisepuede.model_attributes.get_dimensional_attribute_table(
    sisepuede.key_strategy,
)

In [ ]:
sisepuede.model_attributes.get_dimensional_attribute_table(
    sisepuede.key_design,
)

In [ ]:
# ?sisepuede.project_scenarios

In [ ]:
# project across 2 futures for 1 design and, notably, *all* strategies (no filtering)
dict_filt = {
    "future_id": [0],
    "design_id": [0],
    "strategy_id": [1020, 2012, 3025, 4006, 5009, 6002] # note that 5009 does not exist in the attribute table. 
}

primary_keys_out = sisepuede(
    dict_filt,
    chunk_size = 2 # how often do we write to the output database
)

# After running the model, we can see the primary ids that completed successfully

In [ ]:
for k, v in primary_keys_out.items():
    print(v)

##  Before we read in the data, a little on primary ids
- Primary ids are stored in a virtual table (indexing mechanism, the `sisepuede.odpt_primary` object)
- You can find the primary id using `sisepuede.odpt_primary.get_key_value`

In [ ]:
sisepuede.odpt_primary.get_key_value(
    **{
        "design_id": 0,
        "future_id": 0,
        "strategy_id": 1014
    }
)



##  You can quickly see what the dimensions associated with each primary key are by calling:

###  `sisepuede.odpt_primary.get_indexing_dataframe_from_primary_key`


In [ ]:
sisepuede.odpt_primary.get_indexing_dataframe_from_primary_key(
    list(primary_keys_out.get(_REGION_NAME))
)

In [ ]:
sisepuede.key_time_period

###  Retrieve outputs using SISEPUEDE.read_output()

In [ ]:
df_out = sisepuede.read_output(None)
df_out[
    df_out[sisepuede.key_time_period].isin([35])
][
    [sisepuede.key_primary] + [
        x for x in df_out.columns if x.startswith("emission_co2e_subsector")
    ]
]

####  Note that SISEPUEDE.read_output() can be used to execute additional filtering queries by using dict_subset = {}
- e.g., if we only care about the final time period, we can set
    ``dict_subset = {"time_period": 35}``

In [ ]:
sisepuede.read_output(
    None, 
    dict_subset = {
        sisepuede.key_time_period: [35],
        sisepuede.key_region: [_REGION_NAME]
    }
)

###  1. We can examine aggregate emissions across scenarios

In [ ]:
# could query this, but our dataset is small enough to store in memory
df_out_region = (
    df_out[
        df_out[sisepuede.key_region].isin([_REGION_NAME])
    ]
    .reset_index(drop = True)
)

In [ ]:
sisepuede.odpt_primary.get_indexing_dataframe_from_primary_key(
    list(primary_keys_out.get(_REGION_NAME))
)

In [ ]:
strategies.get_strategy(1020).name

In [ ]:
fig, ax = plt.subplots(figsize = (12, 12))
ax.set_xlabel(sisepuede.key_time_period)
ax.set_ylabel("Emission MT CO2e")
# 
all_primaries = sorted(list(df_out_region[sisepuede.key_primary].unique()))
dfg = df_out_region.groupby([sisepuede.key_primary])

for i, df_cur in dfg:

    if i[0] == 60060: continue
    
    y = np.array(df_cur[
        [x for x in df_out.columns if x.startswith("emission_co2e_subsector_")]
    ].sum(axis = 1)) 
    x = np.array(df_cur[sisepuede.key_time_period])
    
    ax.plot(x, y, label = i)
    

ax.legend()


##  Oops! Missing baseline
- Baseline strategy is always 0
- We can re-run sisepuede, and if we include and runs that have already been run, the db management system will avoid re-running those

In [ ]:
# project across 2 futures for 1 design and, notably, *all* strategies (no filtering)
dict_filt = {
    "future_id": [0],
    "design_id": [0],
    "strategy_id": [0, 1020, 6002] 
}

primary_keys_out_2 = sisepuede(
    dict_filt,
    chunk_size = 2 # how often do we write to the output database
)

##  Have to read the output again

In [ ]:
df_out = sisepuede.read_output(None)

# could query this, but our dataset is small enough to store in memory
df_out_region = (
    df_out[
        df_out[sisepuede.key_region].isin([_REGION_NAME])
    ]
    .reset_index(drop = True)
)

df_out_region

###  2. We can also see CO2e by subsector

###  Look at primary ids

In [ ]:
primary_keys = primary_keys_out.get(_REGION_NAME)
sisepuede.odpt_primary.get_indexing_dataframe_from_primary_key(
    sorted(df_out[sisepuede.key_primary].unique())
)


In [ ]:
?sisepuede.odpt_primary.get_key_value

###  Examine baseline strategy

In [ ]:
fig, ax = plt.subplots(figsize = (12, 7))
ax.set_xlabel(sisepuede.key_time_period)
ax.set_ylabel("Emission MT CO2e")

# get scenario key from dictionary
primary_id = sisepuede.odpt_primary.get_key_value(
    **{
        sisepuede.key_design: 0,
        sisepuede.key_future: 0,
        sisepuede.key_strategy: 1020,
    }
)
df_cur = df_out_region[df_out_region[sisepuede.key_primary].isin([primary_id])]

spp.plot_emissions_stack(
    df_cur,
    sisepuede.model_attributes,
    figtuple = (fig, ax), 
)

#ax.legend()



## How about strategy 6002?

In [ ]:
strategies.get_strategy(6002).name


In [ ]:
fig, ax = plt.subplots(figsize = (12, 7))
ax.set_xlabel(sisepuede.key_time_period)
ax.set_ylabel("Emission MT CO2e")

# get scenario key from dictionary
primary_id = sisepuede.odpt_primary.get_key_value(
    **{
        sisepuede.key_design: 0,
        sisepuede.key_future: 0,
        sisepuede.key_strategy: 6002,
    }
)
df_cur = df_out_region[df_out_region[sisepuede.key_primary].isin([primary_id])]

spp.plot_emissions_stack(
    df_cur,
    sisepuede.model_attributes,
    figtuple = (fig, ax), 
)



###  3a. We can go further down and look at gas within subsector - CH4 emisions from Livestock Enteric Fermentation

In [ ]:
# look for baseline
primary_id = sisepuede.odpt_primary.get_key_value(
    **{
        sisepuede.key_design: 0,
        sisepuede.key_future: 0,
        sisepuede.key_strategy: 1020
    }
)

In [ ]:
# variable to examine
modvar = sisepuede.model_attributes.get_variable(
    ":math:\\text{CH}_4 Emissions from Livestock Enteric Fermentation",
    #"Land Use Area"
)


fig, ax = plt.subplots(figsize = (12, 7))
ax.set_xlabel(sisepuede.key_time_period)
ax.set_ylabel("Emission MT CO2e")


df_cur = df_out_region[df_out_region[sisepuede.key_primary].isin([primary_id])]

sup.plot_stack(
    df_cur,
    modvar.fields,
    figtuple = (fig, ax, ),
)

ax.legend()

###  3b. HFCs and PFCs in IPPU

In [ ]:
fig, ax = plt.subplots(figsize = (12, 7))
ax.set_xlabel(sisepuede.key_time_period)
ax.set_ylabel("Emission MT CO2e")


fields = [
    x for x in df_cur.columns 
    if x.startswith("emission_co2e_") and ("_ippu_" in x) and (("hfcs" in x) or ("pfcs" in x))
]


sup.plot_stack(
    df_cur,
    fields,
    figtuple = (fig, ax, ),
)

ax.legend()

###  4. The land use reallocation factor affects land use, imports, and exports

In [ ]:
for ind in [0, 4]:
    i = primary_keys_out[ind]
    fig, ax = plt.subplots(figsize = (12, 12))
    
    df_cur = df_out[df_out["primary_id"] == i]
    df_cur = sisepuede.model_attributes.extract_model_variable(df_cur, "Land Use Area", return_type = "data_frame")
    
    val = sisepuede.generate_scenario_database_from_primary_key(i)["DEMO"]["lndu_reallocation_factor"].iloc[-1]
    
    df_cur.plot.area(ax = ax)
    ax.set_title(f"lambda at 35: {val}")
    
    plt.show()
    
    

###  5. Database functionality streamlines parallelization and analysis

##  Suppose you want to generate a certain future using a primary id


In [ ]:
p_id = sisepuede.odpt_primary.get_key_value(
    **{
        "design_id": 0,
        "future_id": 0,
        "strategy_id": 6002
    }
)

print(f"{sisepuede.key_primary} = {p_id}")

# You can generate tables on the fly using primary ids
- e.g., say you want to see what the input associated with future_id 145, strategy 6002, design 0 looks like
- You could also feed `generate_scenario_database_from_primary_key` a dictionary of dimensional values

In [ ]:

# create a dictionary of runs since different regions will have different dbs
dict_db = sisepuede.generate_scenario_database_from_primary_key(p_id);
df_pid = dict_db.get("peru")

# look at the dataframe
df_pid.head()

In [ ]:
inds = sisepuede.database.db.dict_iterative_database_tables.get("MODEL_OUTPUT").available_indices
inds

In [ ]:
sisepuede.odpt_primary.get_indexing_dataframe_from_primary_key([x[1] for x in sorted(list(inds))])